[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/octonions/octonions.ipynb)

# Octonions and their symmetries

The real numbers, the complex numbers, the quaternions and the octonions, with one, two, four and eight components, are the only number systems in which the length of a product is the product of the lengths, so that any nonzero number can be divided by. Each step up gives something away. The complex numbers can no longer be put in order; the quaternions also lose commutativity, so `a * b` and `b * a` differ; and the octonions lose associativity as well, so that in a product of three, `(a * b) * d` and `a * (b * d)` differ too.

Every geometric algebra is associative, so the octonions are not one; but they can be built from one. Here an octonion is a scalar plus a vector in the geometric algebra of seven directions that square to minus one. To multiply two of them, take their geometric product, multiply it by one minus a fixed trivector that encodes the seven multiplication triples, and keep the scalar and vector parts. Leaving one factor open then turns multiplication into a map on the eight components, and the turns of the seven directions, a family of twenty-one dimensions, contain a family of fourteen that keep the product unchanged: the exceptional group $G_2$. The same octonions also fit, whole, into the eight components of the algebra of three-dimensional space.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import VGA3D as space
from examples.animation import save_figure
from examples.math.octonions import render

In [ ]:
ga = Algebra("a-b-c-d-e-f-g-")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Octonion = ga.gatype.from_blades("1 a b c d e f g")
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
# An octonion as a whole multivector of space, Cl(3, 0).
SpaceOctonion = space.gatype.from_blades("1 x y z yz zx xy xyz")

## 1. A different product on eight components

Keep the scalar and seven vectors as the octonion components. Seven oriented triples specify the multiplication cycles: `a` times `b` is `c`, `b` times `c` is `a`, and reversing the order changes the sign. Together they form one trivector. Multiply in the Clifford algebra using that trivector, then keep only the scalar and vector parts. This projection belongs to the definition of octonion multiplication; the Clifford product itself stays associative.

Drawn with a point for each basis vector and a line for each triple, the seven triples make the Fano plane, the smallest arrangement of points and lines in which every two points lie on one line and every two lines meet in one point; each arrow runs in the order of the product.

In [ ]:
# Each triple supplies one quaternion cycle; every pair of basis vectors occurs once.
cycles = stack([mv.abc, mv.ade, mv.agf, mv.bdf, mv.beg, mv.cdg, mv.cfe])
orientation = cycles.sum(axis=0)
# Two open octonion slots give the whole multiplication in one expression.
multiply = ((Octonion * Octonion) * (1 - orientation)).cast(Octonion)  # [] Octonion <- (Octonion, Octonion)

print("a times b")
print(multiply(mv.a, mv.b).coefficient_table())
print("b times a")
print(multiply(mv.b, mv.a).coefficient_table())
save_figure(render.draw_fano(cycles), "octonions");

## 2. Lengths multiply; brackets matter

Grade involution preserves the scalar part and negates the vector part. Taking the scalar product with that involution gives the positive squared length: the sign change compensates for the negative squares of the seven basis vectors. Multiplication preserves that length rule even though two ways of bracketing three factors can disagree. Any two octonions still generate an associative subalgebra; the failure needs three independent choices.

In [ ]:
first = mv.scalar() + 2 * mv.a - mv.d
second = mv.b + mv.f


def norm_squared(value: Octonion) -> Scalar:
    return value.involute().scalar_product(value)


def associator(first: Octonion, second: Octonion, third: Octonion) -> Octonion:
    return multiply(multiply(first, second), third) - multiply(first, multiply(second, third))


print("Squared length of the product")
print(norm_squared(multiply(first, second)).coefficient_table())
print("Product of the squared lengths")
print((norm_squared(first) * norm_squared(second)).coefficient_table())
print("(a b) d")
print(multiply(multiply(mv.a, mv.b), mv.d).coefficient_table())
print("a (b d)")
print(multiply(mv.a, multiply(mv.b, mv.d)).coefficient_table())

## 3. Leave one factor open

Multiplication by `a` is a map on octonions. Applied twice, it is minus the identity; multiplication by distinct basis vectors anticommutes as maps. These seven maps therefore give a representation of Cl(0,7) on the eight octonion components. Composing maps is associative, although composing multiplication by `a` and then by `b` need not equal multiplication by their octonion product.

In [ ]:
left_a = multiply(mv.a, Octonion)                                   # [] Octonion <- Octonion
left_b = multiply(mv.b, Octonion)
# Composition keeps its brackets: this applies the inner multiplication first.
square = left_a(left_a)
anticommutator = left_a(left_b) + left_b(left_a)

print("Left multiplication by a")
print(left_a.coefficient_table())
print("Its square")
print(square.coefficient_table())
print("Composed with multiplication by b, both ways round, and summed")
print(anticommutator.coefficient_table())

## 4. Which rotations preserve multiplication?

A multiplication-preserving transformation fixes the scalar unit and preserves length, so look among rotations of the seven-dimensional vector space. Infinitesimally, it must obey the product rule: acting on a product must equal acting on its two factors in turn and adding the results.

Clifford multiplication and grade projection already commute with rotations. The only extra ingredient in `multiply` is the fixed trivector, so the product rule reduces to leaving that trivector stationary. A bivector acts by its commutator: leaving the bivector open gives a linear constraint. Its zero singular directions are the allowed generators.

In [ ]:
tolerance = 1e-10
seed = mv.ab

constraint = Bivector.commutator(orientation)                       # [] Trivector <- Bivector
_, singular_values, directions = constraint.svd()
generators: Bivector = directions[singular_values < tolerance]       # [generators] Bivector
print(f"{len(generators)} of {len(directions)} rotation generators preserve multiplication.")

# Project an ordinary rotation onto that space, summing its components over the batch.
symmetry: Bivector = (generators * generators.reverse().scalar_product(seed)).sum(axis=0)
derivation = symmetry.commutator(Octonion)                          # [] Vector <- Octonion
# Acting on a product equals acting on each factor in turn: this binary map vanishes.
product_rule = (derivation(multiply) - multiply(derivation, Octonion)
                - multiply(Octonion, derivation))
with np.printoptions(precision=3):
    print("An infinitesimal symmetry")
    print(derivation.coefficient_table())

The fourteen directions form the Lie algebra of the exceptional group $G_2$. Their commutators stay in the same space. Exponentiating one gives a finite rotation that preserves octonion multiplication; an arbitrary rotation usually does not.

In [ ]:
angle = 0.8

rotations: Rotor = (stack([seed, symmetry]) * (angle / 2)).exp()       # [cases] Rotor
# Compare moving the product with multiplying the moved factors.
rotated_product = rotations >> multiply(mv.a, mv.d)
product_of_rotated = multiply(rotations >> mv.a, rotations >> mv.d)
difference = rotated_product - product_of_rotated
with np.printoptions(precision=3):
    print("Ordinary rotation: difference")
    print(difference[0].coefficient_table())
    print("G₂ rotation: difference")
    print(difference[1].coefficient_table())

## 5. The same octonions in the algebra of space

The eight components of the geometric algebra of three-dimensional space hold an octonion whole, by doubling the quaternions. The even part of a multivector of space, a scalar plus a bivector, is a quaternion; the odd part, a vector plus a trivector, is the pseudoscalar times a second quaternion, kept apart from the first because the pseudoscalar commutes with everything. The product multiplies the even parts as quaternions and brings the odd parts in reversed and in swapped order, `a * c - ~d * b + d * a + b * ~c`, and the length is the scalar part of a multivector against its reverse. Choosing x, y and z for `a`, `b` and `d` fixes the rest through the triples: `c` is `x ⋆ y`, and so on. With every image squaring to minus one, every two anticommuting and all seven triples holding, these are the octonions of the sections above. With z squaring to minus one instead, the same product gives the split octonions, in which a number other than zero can have length zero, and multiply its own conjugate to zero. The conjugate keeps the scalar and negates the rest, as for the complex numbers and the quaternions: numga's `scalar_negation`.

In the notation of Cayley and Dickson an octonion is a pair of quaternions, $(p, q)(r, s) = (pr - \bar s q,\ sp + q \bar r)$; the second of each pair is the odd part here.

In [ ]:
space_mv = NumpyContext(space).multivector


def doubled(first: SpaceOctonion, second: SpaceOctonion) -> SpaceOctonion:
    """The octonion product of two multivectors of space, by doubling the quaternions."""
    # The even parts are quaternions; the odd parts, the pseudoscalar times a second quaternion apiece.
    a, b = first.restrict.even(), first.restrict.odd()
    c, d = second.restrict.even(), second.restrict.odd()
    return a * c - ~d * b + d * a + b * ~c


# x, y and z stand for a, b and d; the triples abc, ade, bdf and cdg make c, e, f and g their products.
a, b, d = space_mv.x, space_mv.y, space_mv.z
c, e, f = doubled(a, b), doubled(a, d), doubled(b, d)
g = doubled(c, d)
images = stack([a, b, c, d, e, f, g])                                         # [images] SpaceOctonion
# The other three triples, agf, beg and cfe, then hold by themselves.
print("a g - f:", render.largest(doubled(a, g) - f), " b e - g:", render.largest(doubled(b, e) - g),
      " c f - e:", render.largest(doubled(c, f) - e))

split = Algebra("x+y+z-")
split_mv = NumpyContext(split).multivector
null = 1 + split_mv.z                                                          # [] SpaceOctonion, split
print("In the split octonions, the length of 1 + z:", render.scalars(null.scalar_norm_squared()))
print("and 1 + z times its conjugate:", render.largest(doubled(null, null.scalar_negation())))

The Clifford construction and the seven oriented triples follow Lemma 2.5 of [Kauhanen and Orelma, *On the Structure of Octonion Regular Functions*](https://doi.org/10.1007/s00006-019-0983-y). John Baez explains the [Clifford actions](https://math.ucr.edu/home/baez/octonions/node6.html) and the [octonion symmetries and derivations](https://math.ucr.edu/home/baez/octonions/node14.html) in *The Octonions*.

The construction in the algebra of space follows Daniele Corradetti, *Recovering Composition Algebras from 3D Geometric Algebras* (2024), who derived it from Anthony Lasenby's octonions in the even subalgebra of the spacetime algebra, *Some recent results for SU(3) and Octonions within the Geometric Algebra approach to the fundamental forces of nature* (2022); [this answer](https://math.stackexchange.com/a/5037113) lays out how the two meet.

In [ ]:
# checks
# Lengths multiply, and the two bracketings of a, b and d differ by twice g.
assert abs(render.scalars(norm_squared(multiply(first, second)) - norm_squared(first) * norm_squared(second))) < 1e-10
assert render.largest(associator(mv.a, mv.b, mv.d) - 2 * mv.g) < 1e-10
# Multiplication by a squares to minus the identity, and multiplications by a and by b anticommute.
assert render.largest(square + Octonion) < 1e-10 and render.largest(anticommutator) < 1e-10
# Fourteen generators, the symmetry among them obeying the product rule; its rotation keeps the product, the
# ordinary rotation does not.
assert len(generators) == 14 and render.largest(product_rule) < 1e-10
assert render.largest(difference[1]) < 1e-10 and render.largest(difference[0]) > 0.1
# In the algebra of space, every image squares to minus one, every two anticommute, and the three triples not
# used to build them hold.
both_orders = doubled(images[:, None], images[None, :]) + doubled(images[None, :], images[:, None])   # [images, images] SpaceOctonion
apart = ~np.eye(images.shape[0], dtype=bool)
assert render.largest(doubled(images, images) + 1) < 1e-10 and render.largest(both_orders[apart]) < 1e-10
assert max(render.largest(doubled(a, g) - f), render.largest(doubled(b, e) - g), render.largest(doubled(c, f) - e)) < 1e-10
# In the split octonions, 1 + z has length zero and multiplies its conjugate to zero.
assert render.largest(null.scalar_norm_squared()) < 1e-10 and render.largest(doubled(null, null.scalar_negation())) < 1e-10